# Trabajo Práctico — Decision Tree vs Random Forest

**Inteligencia Artificial para Datos** · UTN F.R.Re. · Ciclo lectivo 2026

**Dataset:** `data_cardiovascular_risk.csv` — Framingham Heart Study, riesgo de
enfermedad coronaria a 10 años (`TenYearCHD`).

Este notebook es el entregable ejecutable: hace el pre-procesamiento, entrena
las implementaciones **propias** de Decision Tree (C4.5) y Random Forest,
compara su rendimiento bajo distintas parametrizaciones y expone una interfaz
para correr pruebas fácilmente.

### Contenido

0. [Entorno](#0) · 1. [Carga](#1) · 2. [Auditoría inicial](#2) ·
3. [Pre-procesamiento](#3) · 4. [Decision Tree](#4) · 5. [Random Forest](#5)
6. [Validación vs. scikit-learn](#6) · 7. [Protocolo](#7) ·
8. [Barridos DT](#8) · 9. [Barridos RF](#9) · 10. [Ablaciones](#10) ·
11. [Comparación final](#11) · 12. [Interfaz](#12) · 13. [Conclusiones](#13)

> **Cómo ejecutarlo en Colab:** abrirlo desde GitHub con el badge o subir
> solamente este notebook. La primera celda descarga automáticamente los
> cuatro módulos y el CSV desde el repositorio, y sólo si faltan. La subida
> manual queda como respaldo si no hay acceso a la red.

<a id="0"></a>
## 0. Entorno y dependencias

Todo corre en **Google Colab** sin instalaciones extra (numpy, pandas,
matplotlib, scikit-learn e ipywidgets vienen preinstalados). scikit-learn se
usa **sólo** para métricas y como control de validación: los algoritmos están
implementados desde cero en `dt_rf.py`.

La celda siguiente es un *bootstrap* idempotente: revisa si `dt_rf.py`,
`preprocessing.py`, `experiments.py`, `plots.py` y
`data_cardiovascular_risk.csv` están en el directorio de trabajo y descarga
de GitHub los que falten. Si Colab ya clonó el repositorio, no descarga nada.
La subida manual se usa únicamente como respaldo.


In [ ]:
# Bootstrap idempotente: traer los archivos que falten desde el repositorio.
# Si el notebook se abrió desde GitHub (Colab clona el repo) o los archivos ya
# están en /content, esta celda no descarga nada.
import os
import urllib.request

# Versión congelada para que el notebook sea reproducible. Actualizar este
# hash si se modifican los módulos .py o el CSV en el repositorio.
COMMIT = "fa6c4af77435b6e3d6f62bd97a93e636570b9e76"
BASE = (f"https://raw.githubusercontent.com/kahnaisehC/"
        f"analisis-riesgo-cardiovascular/{COMMIT}")
ARCHIVOS = [
    "dt_rf.py",
    "preprocessing.py",
    "experiments.py",
    "plots.py",
    "data_cardiovascular_risk.csv",
]

for nombre in ARCHIVOS:
    if os.path.exists(nombre):
        continue
    try:
        urllib.request.urlretrieve(f"{BASE}/{nombre}", nombre)
        print(f"↓ Descargado desde GitHub: {nombre}")
    except Exception as exc:
        if os.path.exists(nombre):
            os.remove(nombre)
        print(f"⚠ No se pudo descargar {nombre}: {exc}")

print("Listos:", [a for a in ARCHIVOS if os.path.exists(a)])

In [ ]:
import json
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# --- ¿Estamos en Colab? ----------------------------------------------------
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

# --- Asegurar que los módulos del TP estén disponibles ---------------------
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

try:
    import dt_rf, preprocessing, experiments, plots
except ModuleNotFoundError:
    if EN_COLAB:
        from google.colab import files
        print("No se pudieron descargar los módulos del TP (¿sin red?). "
              "Subí manualmente dt_rf.py, preprocessing.py, experiments.py "
              "y plots.py:")
        files.upload()
    import dt_rf, preprocessing, experiments, plots

from dt_rf import DecisionTree, RandomForest
import experiments as E
from plots import draw_tree, plot_confusion_matrix, plot_roc_pr, plot_curve, plot_importances

from IPython.display import display, Markdown

SEED = 42
N_SPLITS = 5
RUTA_DATOS = "data_cardiovascular_risk.csv"
RUTA_PROCESADO = "data/dataset_procesado.csv"
os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("Colab:", EN_COLAB)
print("Versiones:", np.__version__, pd.__version__)

<a id="1"></a>
## 1. Carga del dataset

El dataset tiene 3.390 registros de pacientes, 15 atributos clínicos + un
identificador, y la variable objetivo `TenYearCHD` (1 = desarrolló enfermedad
coronaria en 10 años).

In [ ]:
if not os.path.exists(RUTA_DATOS):
    if EN_COLAB:
        from google.colab import files
        print("No se pudo descargar el CSV (¿sin red?). "
              "Subí manualmente data_cardiovascular_risk.csv:")
        subidos = files.upload()
        RUTA_DATOS = list(subidos)[0]
    else:
        raise FileNotFoundError(
            f"No se encontró {RUTA_DATOS}. Copiá el CSV al directorio de trabajo."
        )

df_raw = preprocessing.load_raw(RUTA_DATOS)
print(f"Dataset: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas")
display(df_raw.head())

<a id="2"></a>
## 2. Auditoría inicial del dataset

Antes de tocar nada: nulos, duplicados, outliers, tipos y anomalías lógicas.
Esta salida es la que se transcribe en la sección **"Informe inicial"** del
informe.

In [ ]:
info = preprocessing.audit(df_raw)
display(Markdown(preprocessing.audit_to_markdown(info)))

In [ ]:
# Visualización de nulos y balance de clase
faltantes = info["missing"]["pct"]
faltantes = faltantes[faltantes > 0].to_dict()

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plots.plot_missing(faltantes, ax=axs[0])
plots.plot_target_balance(df_raw["TenYearCHD"].to_numpy(), ax=axs[1])
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de las variables continuas (para ver colas y outliers)
continuas = ["age", "cigsPerDay", "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose"]
df_raw[continuas].describe().T.round(2)

<a id="3"></a>
## 3. Pre-procesamiento

Decisiones (todas quedan registradas con su conteo en el informe final):

| Paso | Decisión | Motivo |
|---|---|---|
| `id` | se elimina | identificador, sin poder predictivo |
| Duplicados | 0 detectados | se verifica y se reporta |
| Nulos | **imputar**, no borrar filas | borrar las 463 filas con algún nulo eliminaría 67 positivos de una clase que ya es 15 % |
| `BPMeds` | regla: 0 si `prevalentHyp = 0` | el nulo es estructural (no toma medicación) |
| Continuas | MICE (`IterativeImputer`) | preserva correlaciones entre presión, colesterol, glucosa y BMI |
| Outliers | **winsorizar p1–p99** | muchos son valores clínicos reales (glucosa 394, sysBP 295) que pertenecen a la clase de riesgo |
| `age`, `cigsPerDay` | **no** se recortan | sus colas son informativas: mayores y fumadores empedernidos son la población de riesgo |
| `is_smoking` | se conserva (se prueba quitarla en la ablación) | es redundante con `cigsPerDay` |
| Nuevo atributo | `pulsePressure = sysBP - diaBP` | proxy de rigidez arterial, muy usado en riesgo cardiovascular |

In [ ]:
prep = preprocessing.Preprocessor(
    imputer="mice",
    clip_outliers=True,
    clip_lower=0.01,
    clip_upper=0.99,
    drop_is_smoking=False,
    add_pulse_pressure=True,
)
df = prep.fit_transform(df_raw)
df.to_csv(RUTA_PROCESADO, index=False)

display(Markdown(prep.report_markdown()))
print(f"\nDataset procesado guardado en {RUTA_PROCESADO}: {df.shape}")
display(df.head())

In [ ]:
# Controles de calidad sobre el dataset limpio
assert df.isna().sum().sum() == 0, "quedaron nulos"
assert df["TenYearCHD"].isin([0, 1]).all()
assert "id" not in df.columns, "el id debería haberse eliminado"
print("Sin nulos. Clases:", dict(df["TenYearCHD"].value_counts()))
print("Columnas:", list(df.columns))

X = df.drop(columns=["TenYearCHD"])
y = df["TenYearCHD"].to_numpy()

from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
print(f"Train: {X_tr.shape}  Test: {X_te.shape}")

<a id="4"></a>
## 4. Decision Tree (C4.5) — implementación propia

`DecisionTree` (en `dt_rf.py`) implementa:

- **Criterios**: entropía con *gain ratio* (C4.5) o Gini.
- **Splits numéricos**: umbral `≤` elegido por barrido ordenado O(n log n).
- **Splits categóricos**: multi-vía, una rama por categoría.
- **Parada**: `max_depth`, `min_samples_split`, `min_samples_leaf`,
  `min_impurity_decrease`.
- **Poda**: pesimista de C4.5 y *Reduced Error Pruning* con validación.
- **Desbalance**: `class_weight="balanced"`.
- **Importancia de atributos**: decremento ponderado de impureza.

Los atributos categóricos se detectan automáticamente desde los dtypes del
DataFrame (`education`, `sex`, `is_smoking`, `BPMeds`, `prevalent*`, `diabetes`).

In [ ]:
arbol = DecisionTree(
    criterion="gini",       # entropía + gain ratio también disponible
    max_depth=4,
    min_samples_leaf=20,
    class_weight=None,
    random_state=SEED,
)
t0 = time.perf_counter()
arbol.fit(X_tr, y_tr)
print(f"Entrenado en {time.perf_counter() - t0:.3f} s")
print(f"Hojas: {arbol.get_n_leaves()}  |  Profundidad: {arbol.get_depth()}")

proba_arbol = arbol.predict_proba(X_te)[:, 1]
res_arbol = E.evaluate_predictions(y_te, proba_arbol, arbol.predict(X_te))
pd.Series(res_arbol).round(4).to_frame("valor")

In [ ]:
print(arbol.export_text(list(X.columns)))

In [ ]:
# Dibujo del árbol (las primeras 3 capas)
draw_tree(arbol, list(X.columns), max_depth=3)
plt.show()

<a id="5"></a>
## 5. Random Forest — implementación propia

`RandomForest` combina:

- **Bagging**: cada árbol entrena con una muestra bootstrap.
- **Subespacios aleatorios**: en cada split se evalúa sólo un subconjunto de
  atributos (`max_features="sqrt"` por defecto, el valor clásico de Breiman).
- **Voto**: suave (promedio de probabilidades) o duro (mayoría).
- **OOB**: exactitud *out-of-bag*, estimación gratuita del error de
  generalización (cada muestra se predice con los árboles que no la vieron).

In [ ]:
bosque = RandomForest(
    n_estimators=100,
    criterion="gini",
    max_depth=8,
    max_features="sqrt",
    voting="soft",
    class_weight=None,
    n_jobs=-1,
    random_state=SEED,
)
t0 = time.perf_counter()
bosque.fit(X_tr, y_tr)
print(f"Entrenado en {time.perf_counter() - t0:.2f} s")
print(f"OOB accuracy: {bosque.oob_score_:.4f}")
print(f"Hojas promedio por árbol: {np.mean([t.get_n_leaves() for t in bosque.estimators_]):.1f}")

proba_bosque = bosque.predict_proba(X_te)[:, 1]
res_bosque = E.evaluate_predictions(y_te, proba_bosque, bosque.predict(X_te))
pd.Series(res_bosque).round(4).to_frame("valor")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plot_importances(bosque.feature_importances_, list(X.columns), top_n=10, ax=axs[0],
                 title="Random Forest")
plot_importances(arbol.feature_importances_, list(X.columns), top_n=10, ax=axs[1],
                 title="Decision Tree")
plt.tight_layout()
plt.show()

<a id="6"></a>
## 6. Validación de las implementaciones contra scikit-learn

Antes de sacar conclusiones, se verifica que las implementaciones propias
reproducen a las de scikit-learn con la misma configuración. Es un *sanity
check*: si las métricas quedan cerca, el código está bien.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

filas = []
for nombre, modelo in [
    ("DT propio (entropía + gain ratio)", DecisionTree(max_depth=5)),
    ("DT propio (entropía + gain)", DecisionTree(max_depth=5, split_metric="gain")),
    ("DT propio (gini)", DecisionTree(max_depth=5, criterion="gini")),
    ("DT sklearn (entropy)", DecisionTreeClassifier(max_depth=5, criterion="entropy", random_state=0)),
    ("DT sklearn (gini)", DecisionTreeClassifier(max_depth=5, criterion="gini", random_state=0)),
    ("RF propio", RandomForest(n_estimators=100, max_depth=8, criterion="gini", n_jobs=-1, random_state=0)),
    ("RF sklearn", RandomForestClassifier(n_estimators=100, max_depth=8, random_state=0, n_jobs=-1)),
]:
    t0 = time.perf_counter()
    modelo.fit(X_tr, y_tr)
    t = time.perf_counter() - t0
    proba = modelo.predict_proba(X_te)[:, 1]
    r = E.evaluate_predictions(y_te, proba, modelo.predict(X_te))
    filas.append({
        "modelo": nombre,
        "accuracy": round(r["accuracy"], 4),
        "f1": round(r["f1"], 4),
        "roc_auc": round(r["roc_auc"], 4),
        "tiempo (s)": round(t, 3),
    })

comparacion_sklearn = pd.DataFrame(filas)
comparacion_sklearn.to_csv("results/comparacion_sklearn.csv", index=False)
comparacion_sklearn

<a id="7"></a>
## 7. Protocolo experimental

- **Validación cruzada estratificada 5-fold** (media ± desvío entre folds).
- **Hold-out 70/30** para matrices de confusión y curvas (un solo corte).
- **Métricas**: accuracy, balanced accuracy, precision, recall, F1,
  ROC-AUC y PR-AUC.
- **F1 óptimo**: umbral que maximiza F1 (con 15 % de positivos, el umbral 0,5
  casi nunca predice la clase 1 y el F1 se vuelve engañoso).

> Un clasificador trivial que diga siempre "no" acierta ≈ **84,9 %**. Por eso
> la accuracy sola no alcanza: se reportan balanced accuracy, recall y PR-AUC.

In [ ]:
def mostrar(res):
    '''Resumen legible de un resultado de cross_validate.'''
    return pd.DataFrame({
        "métrica": E.METRICS,
        "media": [res[m] for m in E.METRICS],
        "desvío": [res[f"{m}_std"] for m in E.METRICS],
    }).round(4)

<a id="8"></a>
## 8. Barrido de hiperparámetros — Decision Tree

Se estudia cómo cambia el rendimiento con la profundidad, el mínimo de
muestras por hoja, el criterio de impureza y la poda.

In [ ]:
t0 = time.perf_counter()
barrido_prof = E.sweep(
    E.make_dt, "max_depth", [1, 2, 3, 4, 5, 6, 8, 10, 12, 16],
    X, y, n_splits=N_SPLITS, fixed={"criterion": "gini"}, verbose=False,
)
print(f"{time.perf_counter() - t0:.1f} s")
barrido_prof.to_csv("results/dt_barrido_profundidad.csv", index=False)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plot_curve(barrido_prof["max_depth"], barrido_prof["accuracy"],
           barrido_prof["accuracy_std"], "max_depth", "accuracy",
           "Accuracy vs profundidad", ax=axs[0])
plot_curve(barrido_prof["max_depth"], barrido_prof["roc_auc"],
           barrido_prof["roc_auc_std"], "max_depth", "ROC-AUC",
           "AUC vs profundidad", ax=axs[1])
plt.tight_layout()
plt.show()
barrido_prof[["max_depth", "accuracy", "balanced_accuracy", "f1", "f1_opt",
              "roc_auc", "leaves"]].round(4)

In [ ]:
barrido_hoja = E.sweep(
    E.make_dt, "min_samples_leaf", [1, 2, 5, 10, 20, 40, 80],
    X, y, n_splits=N_SPLITS, fixed={"criterion": "gini", "max_depth": 8},
    verbose=False,
)
barrido_hoja.to_csv("results/dt_barrido_min_hoja.csv", index=False)
barrido_hoja[["min_samples_leaf", "accuracy", "f1_opt", "roc_auc", "leaves"]].round(4)

In [ ]:
# Criterios de impureza y métrica de split (gain ratio de C4.5 vs gain puro)
filas = []
for criterion, split_metric, etiqueta in [
    ("entropy", "gain_ratio", "entropía + gain ratio"),
    ("entropy", "gain", "entropía + gain"),
    ("gini", None, "gini"),
]:
    r = E.cross_validate(
        lambda c=criterion, sm=split_metric: DecisionTree(
            max_depth=6, criterion=c, split_metric=sm
        ),
        X, y, n_splits=N_SPLITS, verbose=False,
    )
    r["criterion"] = criterion
    r["split_metric"] = split_metric
    filas.append(r)
    print(f"{etiqueta:24s} acc={r['accuracy']:.4f} auc={r['roc_auc']:.4f}")
barrido_criterio = pd.DataFrame(filas)
barrido_criterio.to_csv("results/dt_barrido_criterio.csv", index=False)
barrido_criterio[["criterion", "split_metric", "accuracy", "f1_opt",
                  "roc_auc", "leaves"]].round(4)

In [ ]:
# Poda: sin poda, pesimista (C4.5) y REP
filas = []
for etiqueta, poda in [("sin poda", False),
                       ("pesimista (C4.5)", "pessimistic"),
                       ("REP (20 % validación)", "rep")]:
    r = E.cross_validate(lambda: DecisionTree(max_depth=12, criterion="gini"),
                         X, y, n_splits=N_SPLITS, prune=poda, verbose=False)
    r["poda"] = etiqueta
    filas.append(r)

barrido_poda = pd.DataFrame(filas)
barrido_poda.to_csv("results/dt_barrido_poda.csv", index=False)
barrido_poda[["poda", "accuracy", "f1_opt", "roc_auc", "leaves"]].round(4)

In [ ]:
# Efecto del balanceo de clases
filas = []
for etiqueta, cw in [("sin balanceo", None), ("balanced", "balanced")]:
    r = E.cross_validate(
        lambda: DecisionTree(max_depth=6, criterion="gini", class_weight=cw),
        X, y, n_splits=N_SPLITS, verbose=False,
    )
    r["class_weight"] = etiqueta
    filas.append(r)
balanceo_dt = pd.DataFrame(filas)
balanceo_dt.to_csv("results/dt_class_weight.csv", index=False)
balanceo_dt[["class_weight", "accuracy", "balanced_accuracy", "precision",
             "recall", "f1"]].round(4)

<a id="9"></a>
## 9. Barrido de hiperparámetros — Random Forest

Cantidad de árboles (con curva OOB), atributos por split, profundidad y tipo
de voto.

In [ ]:
t0 = time.perf_counter()
barrido_arboles = E.sweep(
    E.make_rf, "n_estimators", [10, 25, 50, 100, 200, 300],
    X, y, n_splits=N_SPLITS,
    fixed={"max_depth": 12, "criterion": "gini", "n_jobs": -1},
    verbose=False,
)
print(f"{time.perf_counter() - t0:.1f} s")
barrido_arboles.to_csv("results/rf_barrido_arboles.csv", index=False)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plot_curve(barrido_arboles["n_estimators"], barrido_arboles["roc_auc"],
           barrido_arboles["roc_auc_std"], "n_estimators", "ROC-AUC",
           "AUC vs cantidad de árboles", ax=axs[0])
plot_curve(barrido_arboles["n_estimators"], barrido_arboles["oob_accuracy"],
           barrido_arboles["oob_accuracy_std"], "n_estimators", "OOB accuracy",
           "Error OOB vs cantidad de árboles", ax=axs[1])
plt.tight_layout()
plt.show()
barrido_arboles[["n_estimators", "accuracy", "f1_opt", "roc_auc",
                 "oob_accuracy", "fit_time"]].round(4)

In [ ]:
barrido_mf = E.sweep(
    E.make_rf, "max_features", ["sqrt", "log2", 0.3, 0.6],
    X, y, n_splits=N_SPLITS,
    fixed={"n_estimators": 100, "max_depth": 12, "criterion": "gini", "n_jobs": -1},
    verbose=False,
)
barrido_mf.to_csv("results/rf_barrido_max_features.csv", index=False)
barrido_mf[["max_features", "accuracy", "f1_opt", "roc_auc", "oob_accuracy"]].round(4)

In [ ]:
barrido_prof_rf = E.sweep(
    E.make_rf, "max_depth", [3, 5, 8, 12, 20, None],
    X, y, n_splits=N_SPLITS,
    fixed={"n_estimators": 100, "max_features": "sqrt", "criterion": "gini",
           "n_jobs": -1},
    verbose=False,
)
barrido_prof_rf.to_csv("results/rf_barrido_profundidad.csv", index=False)
barrido_prof_rf[["max_depth", "accuracy", "f1_opt", "roc_auc",
                 "oob_accuracy"]].round(4)

In [ ]:
filas = []
for voto in ["soft", "hard"]:
    r = E.cross_validate(
        lambda: RandomForest(n_estimators=100, max_depth=12, criterion="gini",
                             voting=voto, n_jobs=-1),
        X, y, n_splits=N_SPLITS, verbose=False,
    )
    r["voting"] = voto
    filas.append(r)
votacion = pd.DataFrame(filas)
votacion.to_csv("results/rf_voting.csv", index=False)
votacion[["voting", "accuracy", "f1_opt", "roc_auc"]].round(4)

<a id="10"></a>
## 10. Ablaciones de pre-procesamiento

Cada configuración se evalúa con el pre-procesamiento ajustado **dentro** de
cada fold (sin fuga de información). Se mide el impacto de la estrategia de
imputación, del winsorizado, de `is_smoking` y de `pulsePressure`.

In [ ]:
configuraciones = [
    ("MICE + winsor", dict(imputer="mice", clip_outliers=True)),
    ("mediana + winsor", dict(imputer="median", clip_outliers=True)),
    ("KNN + winsor", dict(imputer="knn", clip_outliers=True)),
    ("MICE sin winsor", dict(imputer="mice", clip_outliers=False)),
    ("MICE sin is_smoking", dict(imputer="mice", clip_outliers=True, drop_is_smoking=True)),
    ("MICE sin pulsePressure", dict(imputer="mice", clip_outliers=True, add_pulse_pressure=False)),
]

filas = []
for etiqueta, params in configuraciones:
    r = E.cross_validate_with_preprocessing(
        df_raw, "TenYearCHD",
        lambda p=params: preprocessing.Preprocessor(**p),
        lambda: RandomForest(n_estimators=60, max_depth=10, criterion="gini", n_jobs=-1),
        n_splits=3,
    )
    r["configuracion"] = etiqueta
    filas.append(r)
    print(f"{etiqueta:26s} acc={r['accuracy']:.4f} auc={r['roc_auc']:.4f}")

ablaciones = pd.DataFrame(filas)
ablaciones.to_csv("results/ablaciones_preprocesamiento.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(ablaciones["configuracion"], ablaciones["roc_auc"], color="#55A868")
ax.set_xlim(ablaciones["roc_auc"].min() - 0.01, ablaciones["roc_auc"].max() + 0.01)
ax.set_xlabel("ROC-AUC (CV 3-fold)")
ax.set_title("Impacto de las decisiones de pre-procesamiento")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()
ablaciones[["configuracion", "accuracy", "f1_opt", "roc_auc"]].round(4)

<a id="11"></a>
## 11. Comparación final DT vs RF

Se elige la mejor configuración de cada modelo **por ROC-AUC en validación
cruzada** (no por accuracy: con 15 % de positivos, la accuracy premia predecir
siempre la clase mayoritaria) y se comparan sobre el hold-out.

In [ ]:
from itertools import product

# Los barridos univariados no capturan la interacción profundidad × mínimo por
# hoja. Se evalúa una grilla chica para elegir la configuración final del DT.
filas_grilla = []
for prof, hoja in product([4, 5, 6], [1, 5, 20, 40]):
    r = E.cross_validate(
        lambda p=prof, h=hoja: DecisionTree(
            max_depth=p, min_samples_leaf=h, criterion="gini"
        ),
        X, y, n_splits=N_SPLITS, verbose=False,
    )
    filas_grilla.append({"max_depth": prof, "min_samples_leaf": hoja,
                         "roc_auc": r["roc_auc"]})
grilla_dt = pd.DataFrame(filas_grilla)
mejor_dt = grilla_dt.loc[grilla_dt["roc_auc"].idxmax()]

# Misma idea para el bosque: profundidad × max_features.
filas_grilla = []
for prof, mf in product([3, 5, 8], ["sqrt", 0.3]):
    r = E.cross_validate(
        lambda p=prof, m=mf: RandomForest(
            n_estimators=100, max_depth=p, max_features=m,
            criterion="gini", n_jobs=-1,
        ),
        X, y, n_splits=N_SPLITS, verbose=False,
    )
    filas_grilla.append({"max_depth": prof, "max_features": str(mf),
                         "roc_auc": r["roc_auc"]})
grilla_rf = pd.DataFrame(filas_grilla)
mejor_rf = grilla_rf.loc[grilla_rf["roc_auc"].idxmax()]

mejor_arboles = int(barrido_arboles.loc[barrido_arboles["roc_auc"].idxmax(), "n_estimators"])
mejor_mf = mejor_rf["max_features"]
if mejor_mf not in ("sqrt", "log2"):
    mejor_mf = float(mejor_mf)
mejor_prof = int(mejor_dt["max_depth"])
mejor_hoja = int(mejor_dt["min_samples_leaf"])
mejor_prof_rf = int(mejor_rf["max_depth"])

print("Mejor DT :", dict(max_depth=mejor_prof, min_samples_leaf=mejor_hoja, criterion="gini"))
print("Mejor RF :", dict(n_estimators=mejor_arboles, max_depth=mejor_prof_rf,
                        max_features=mejor_mf, criterion="gini"))

config_dt = dict(max_depth=mejor_prof, min_samples_leaf=mejor_hoja, criterion="gini")
config_rf = dict(n_estimators=mejor_arboles, max_depth=mejor_prof_rf,
                 max_features=mejor_mf, criterion="gini", n_jobs=-1)

cv_dt = E.cross_validate(lambda: DecisionTree(**config_dt), X, y, n_splits=N_SPLITS)
cv_rf = E.cross_validate(lambda: RandomForest(**config_rf), X, y, n_splits=N_SPLITS)

tabla_cv = pd.DataFrame([
    {"modelo": "Decision Tree", **{m: cv_dt[m] for m in E.METRICS}, "tiempo (s)": cv_dt["fit_time"]},
    {"modelo": "Random Forest", **{m: cv_rf[m] for m in E.METRICS}, "tiempo (s)": cv_rf["fit_time"]},
]).set_index("modelo")
tabla_cv.round(4)

In [ ]:
# Hold-out 70/30: modelos finales
modelos = {
    "Decision Tree": DecisionTree(**config_dt).fit(X_tr, y_tr),
    "Random Forest": RandomForest(**config_rf).fit(X_tr, y_tr),
}
probas = {n: m.predict_proba(X_te)[:, 1] for n, m in modelos.items()}
preds = {n: m.predict(X_te) for n, m in modelos.items()}

filas = []
for nombre in modelos:
    r = E.evaluate_predictions(y_te, probas[nombre], preds[nombre])
    filas.append({"modelo": nombre, **r,
                  "hojas": modelos[nombre].get_n_leaves()
                  if hasattr(modelos[nombre], "get_n_leaves")
                  else np.mean([t.get_n_leaves() for t in modelos[nombre].estimators_])})
tabla_holdout = pd.DataFrame(filas).set_index("modelo")
tabla_holdout.round(4)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, nombre in zip(axs, modelos):
    cm = E.confusion(y_te, preds[nombre])
    plot_confusion_matrix(cm, labels=("0", "1"), ax=ax, title=f"{nombre} (umbral 0,5)")
plt.tight_layout()
plt.show()

print("F1 con umbral óptimo (el que maximiza F1 en test):")
for nombre in modelos:
    umbral, f1_opt = E.optimal_threshold(y_te, probas[nombre])
    print(f"  {nombre:14s} umbral={umbral:.3f}  F1={f1_opt:.4f}")

In [ ]:
plot_roc_pr(y_te, probas, title_suffix=" (hold-out)")
plt.show()

In [ ]:
# Importancias sobre el hold-out
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
plot_importances(modelos["Decision Tree"].feature_importances_, list(X.columns),
                 top_n=10, ax=axs[0], title="Decision Tree")
plot_importances(modelos["Random Forest"].feature_importances_, list(X.columns),
                 top_n=10, ax=axs[1], title="Random Forest")
plt.tight_layout()
plt.show()

pd.DataFrame({
    "atributo": list(X.columns),
    "DT": modelos["Decision Tree"].feature_importances_.round(4),
    "RF": modelos["Random Forest"].feature_importances_.round(4),
}).sort_values("RF", ascending=False)

In [ ]:
# Comparación estadística sobre las exactitudes por fold (Wilcoxon)
from sklearn.model_selection import StratifiedKFold

def exactitudes_por_fold(make, X, y, n_splits=N_SPLITS):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    out = []
    for k, (tr, te) in enumerate(skf.split(np.zeros(len(y)), y), start=1):
        m = make()
        if getattr(m, "random_state", 0) is None:
            m.set_params(random_state=SEED + k)
        m.fit(E._take(X, tr), y[tr])
        out.append(float((m.predict(E._take(X, te)) == y[te]).mean()))
    return np.array(out)

acc_dt = exactitudes_por_fold(lambda: DecisionTree(**config_dt), X, y)
acc_rf = exactitudes_por_fold(lambda: RandomForest(**config_rf), X, y)
print("DT por fold:", np.round(acc_dt, 4))
print("RF por fold:", np.round(acc_rf, 4))

try:
    from scipy.stats import wilcoxon
    stat, p = wilcoxon(acc_dt, acc_rf)
    print(f"Wilcoxon: estadístico={stat:.3f}, p={p:.4f}")
    print("Con n=5 folds el p mínimo posible es 0,0625: no se puede rechazar H0.")
except Exception as e:
    print("scipy no disponible:", e)

<a id="12"></a>
## 12. Interfaz para correr pruebas

Dos formas de usar el TP:

1. **`run_experiment(...)`**: función programática. Devuelve la tabla de
   métricas y los modelos entrenados (sirve para automatizar o testear).
2. **Panel interactivo**: controles para cambiar la parametrización y ver los
   resultados sin tocar código.

In [ ]:
def run_experiment(
    dt_max_depth=4,
    dt_min_samples_leaf=20,
    dt_criterion="gini",
    dt_class_weight=None,
    rf_n_estimators=100,
    rf_max_depth=8,
    rf_max_features="sqrt",
    rf_voting="soft",
    rf_class_weight=None,
    n_splits=5,
    incluir_graficos=True,
):
    '''Entrena DT y RF con la parametrización dada y devuelve la comparación.

    Returns
    -------
    tabla : DataFrame
        Métricas de CV 5-fold para ambos modelos.
    modelos : dict
        Modelos reentrenados sobre todo el dataset (para inspeccionar).
    '''
    make_dt = lambda: DecisionTree(
        max_depth=dt_max_depth,
        min_samples_leaf=dt_min_samples_leaf,
        criterion=dt_criterion,
        class_weight=dt_class_weight,
    )
    make_rf = lambda: RandomForest(
        n_estimators=rf_n_estimators,
        max_depth=rf_max_depth,
        max_features=rf_max_features,
        voting=rf_voting,
        criterion="gini",
        class_weight=rf_class_weight,
        n_jobs=-1,
    )

    cv = {
        "Decision Tree": E.cross_validate(make_dt, X, y, n_splits=n_splits),
        "Random Forest": E.cross_validate(make_rf, X, y, n_splits=n_splits),
    }
    tabla = pd.DataFrame([
        {"modelo": nombre, **{m: res[m] for m in E.METRICS},
         "tiempo (s)": res["fit_time"]}
        for nombre, res in cv.items()
    ]).set_index("modelo")

    modelos = {
        "Decision Tree": make_dt().fit(X, y),
        "Random Forest": make_rf().fit(X, y),
    }

    if incluir_graficos:
        fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
        plot_importances(modelos["Decision Tree"].feature_importances_, list(X.columns),
                         top_n=8, ax=axs[0], title="DT — importancia")
        plot_importances(modelos["Random Forest"].feature_importances_, list(X.columns),
                         top_n=8, ax=axs[1], title="RF — importancia")
        plt.tight_layout()
        plt.show()

    return tabla, modelos, cv


tabla_ejemplo, modelos_ejemplo, cv_ejemplo = run_experiment()
tabla_ejemplo.round(4)

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    HAY_WIDGETS = True
except ImportError:
    HAY_WIDGETS = False
    print("ipywidgets no disponible. Usar run_experiment(...) directamente.")

if HAY_WIDGETS:
    estilo = {"description_width": "170px"}
    ancho = widgets.Layout(width="420px")

    w_dt_prof = widgets.IntSlider(4, 1, 20, 1, description="DT profundidad:", style=estilo, layout=ancho)
    w_dt_hoja = widgets.IntSlider(20, 1, 80, 1, description="DT mín. por hoja:", style=estilo, layout=ancho)
    w_dt_crit = widgets.Dropdown(options=["gini", "entropy"], value="gini",
                                 description="DT criterio:", style=estilo, layout=ancho)
    w_dt_cw = widgets.Dropdown(options=[("sin balanceo", None), ("balanced", "balanced")],
                               value=None, description="DT class_weight:", style=estilo, layout=ancho)
    w_rf_arb = widgets.IntSlider(100, 10, 300, 10, description="RF árboles:", style=estilo, layout=ancho)
    w_rf_prof = widgets.IntSlider(8, 1, 20, 1, description="RF profundidad:", style=estilo, layout=ancho)
    w_rf_mf = widgets.Dropdown(options=["sqrt", "log2", 0.3, 0.6], value="sqrt",
                               description="RF max_features:", style=estilo, layout=ancho)
    w_rf_voto = widgets.Dropdown(options=["soft", "hard"], value="soft",
                                 description="RF voto:", style=estilo, layout=ancho)
    w_rf_cw = widgets.Dropdown(options=[("sin balanceo", None), ("balanced", "balanced")],
                               value=None, description="RF class_weight:", style=estilo, layout=ancho)
    w_cv = widgets.IntSlider(5, 3, 10, 1, description="Folds CV:", style=estilo, layout=ancho)

    boton = widgets.Button(description="Ejecutar prueba", button_style="success",
                           icon="play", layout=widgets.Layout(width="200px"))
    salida = widgets.Output()

    def _ejecutar(_):
        with salida:
            clear_output(wait=True)
            print("Entrenando...")
            t0 = time.perf_counter()
            tabla, modelos, _ = run_experiment(
                dt_max_depth=w_dt_prof.value,
                dt_min_samples_leaf=w_dt_hoja.value,
                dt_criterion=w_dt_crit.value,
                dt_class_weight=w_dt_cw.value,
                rf_n_estimators=w_rf_arb.value,
                rf_max_depth=w_rf_prof.value,
                rf_max_features=w_rf_mf.value,
                rf_voting=w_rf_voto.value,
                rf_class_weight=w_rf_cw.value,
                n_splits=w_cv.value,
                incluir_graficos=False,
            )
            print(f"Listo en {time.perf_counter() - t0:.1f} s\n")
            display(tabla.round(4))
            dt = modelos["Decision Tree"]
            rf = modelos["Random Forest"]
            print("Modelos reentrenados sobre todo el dataset:")
            print(f"  DT: {dt.get_n_leaves()} hojas, profundidad {dt.get_depth()}")
            print(f"  RF: {len(rf.estimators_)} árboles, OOB accuracy = {rf.oob_score_:.4f}")
            print("El análisis de umbral está en la sección 11.")

    boton.on_click(_ejecutar)
    display(widgets.VBox([
        widgets.HTML("<b>Decision Tree</b>"),
        w_dt_prof, w_dt_hoja, w_dt_crit, w_dt_cw,
        widgets.HTML("<b>Random Forest</b>"),
        w_rf_arb, w_rf_prof, w_rf_mf, w_rf_voto, w_rf_cw,
        widgets.HTML("<b>Validación</b>"),
        w_cv, boton, salida,
    ]))

<a id="13"></a>
## 13. Conclusiones

Se completan en el informe (`informe.md`). Los puntos principales:

1. **El dataset está desbalanceado** (15 % de positivos). La accuracy es
   engañosa: un modelo constante acierta 84,9 %. Hay que mirar balanced
   accuracy, recall, F1 y PR-AUC.
2. **Decision Tree** es rápido (≈0,1 s), interpretable y sirve como línea base,
   pero es inestable: pequeños cambios en los datos cambian el árbol. Sin poda
   sobreajusta (muchas hojas, accuracy de train alta y AUC de test baja).
3. **Random Forest** mejora el AUC de forma consistente (≈+2 puntos en CV y
   +6 en el hold-out) y estabiliza las predicciones, a costa de ~20× más tiempo
   de entrenamiento y de perder interpretabilidad directa (se compensa con
   importancias y OOB).
4. **El gain ratio de C4.5 no siempre es el mejor criterio**: en este dataset
   la entropía con *information gain* y Gini generalizan mejor. Es un hallazgo
   interesante para discutir: el gain ratio penaliza splits con mucha
   entropía de partición y puede elegir cortes que aíslan pocos casos.
5. **Pre-procesamiento**: winsorizar apenas mueve el AUC (los árboles son
   robustos a outliers); imputar con MICE, mediana o KNN da resultados
   equivalentes; `pulsePressure` aporta poco pero no molesta.
6. **Poda**: REP con poca validación es demasiado agresiva en datos
   desbalanceados (colapsa el árbol); la poda pesimista de C4.5 es un mejor
   compromiso entre tamaño y rendimiento.

In [ ]:
# Exportar todo lo necesario para el informe
resumen = {
    "dataset": {
        "filas": int(df_raw.shape[0]),
        "columnas": int(df_raw.shape[1]),
        "nulos_celda": int(info["missing_total_cells"]),
        "filas_con_nulos": int(info["missing_rows_any"]),
        "duplicados": int(info["duplicates"]["exactos"]),
        "positivos": int((df_raw["TenYearCHD"] == 1).sum()),
        "pct_positivos": round(float((df_raw["TenYearCHD"] == 1).mean() * 100), 2),
        "imputados": int(sum(prep.report_["imputados"].values())),
        "winsorizados": int(sum(prep.report_["winsorizados"].values())),
    },
    "config_dt": config_dt,
    "config_rf": {k: (None if v is None else str(v)) for k, v in config_rf.items()},
    "cv_dt": cv_dt,
    "cv_rf": cv_rf,
    "holdout": tabla_holdout.to_dict(),
    "baseline_mayoritaria": round(float((y == 0).mean()), 4),
    "umbral_optimo": {
        n: E.optimal_threshold(y_te, probas[n]) for n in modelos
    },
}
with open("results/resumen_notebook.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False, default=float)

print("Resumen guardado en results/resumen_notebook.json")
print("Tablas en results/*.csv")